# REAL-WORLD WBAN SYBIL DETECTION TEST
## Test the Random Forest Model on Your Real Dataset

This notebook tests the trained Random Forest model from Stage 2 on real-world WBAN dataset to:
1. Detect Sybil nodes vs Normal nodes
2. Show detection percentage
3. List which specific nodes are identified as Sybil
4. Display confidence scores

## Section 1: Import Libraries

In [5]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(" All libraries imported successfully")

 All libraries imported successfully


## Section 2: Load Trained Model and Preprocessing Tools

In [6]:
# Load the trained Random Forest model from Stage 2
try:
    model_path = '../Stage_2_Fast_Models/stage2_random_forest_model.pkl'
    rf_model = pickle.load(open(model_path, 'rb'))
    print(f" Random Forest model loaded successfully")
    print(f"  - Type: {type(rf_model).__name__}")
    print(f"  - Number of trees: {rf_model.n_estimators}")
    print(f"  - Max depth: {rf_model.max_depth}")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    print(f"  Make sure path is correct: {model_path}")
    print(f"  Alternative: Update path to your actual model location")

# Load preprocessing tools from Stage 1
try:
    stage1_data_path = '../Stage_1_Data_Prep/stage1_preprocessed_data.pkl'
    stage1_data = pickle.load(open(stage1_data_path, 'rb'))
    scaler = stage1_data['scaler']
    feature_names = stage1_data['X_columns']
    print(f"\n Preprocessing tools loaded")
    print(f"  - Scaler: StandardScaler")
    print(f"  - Features: {len(feature_names)}")
    print(f"  - Feature names: {feature_names[:5]}...")  # Show first 5
except Exception as e:
    print(f"✗ Error loading preprocessing tools: {e}")
    print(f"  Make sure path is correct: {stage1_data_path}")
    print(f"  Alternative: Update path to your actual data location")

 Random Forest model loaded successfully
  - Type: RandomForestClassifier
  - Number of trees: 300
  - Max depth: 15

 Preprocessing tools loaded
  - Scaler: StandardScaler
  - Features: 19
  - Feature names: ['window_start_s', 'window_end_s', 'pps', 'iat_mean', 'iat_std']...


## Section 3: Load Your Real-World WBAN Test Dataset

**IMPORTANT**: Update the `test_data_path` below with your actual test dataset path.

Your dataset should have:
- Same features as training data (19 features)
- Optional: 'label' or 'target' column for validation
- Optional: 'node_id' or 'mac' column to identify nodes

In [7]:

try:
    df_test = pd.read_csv('test4.csv')
    print(f"✓ Test dataset loaded successfully")
    print(f"\nDataset Summary:")
    print(f"  - Shape: {df_test.shape[0]} samples, {df_test.shape[1]} columns")
    print(f"  - Columns: {list(df_test.columns)}")
    print(f"\nFirst 5 rows:")
    print(df_test.head())
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    print(f"\nPlease update 'test_data_path' to point to your actual dataset.")

✓ Test dataset loaded successfully

Dataset Summary:
  - Shape: 1048575 samples, 20 columns
  - Columns: ['node_id', 'node_mac', 'window_start_s', 'window_end_s', 'pps', 'iat_mean', 'iat_std', 'seq_gap_mean', 'seq_gap_max', 'seq_reset_rate', 'dup_seq_rate', 'out_of_order_rate', 'boot_change_rate', 'udp_pkt_count', 'rssi_mean', 'rssi_std', 'rssi_min', 'rssi_max', 'rssi_frame_count', 'rssi_missing']

First 5 rows:
        node_id           node_mac  window_start_s  window_end_s         pps  \
0        ecg_01  7c:9e:bd:f6:ce:40               0             5    9.927955   
1        eeg_01  34:94:54:aa:79:d0               0             5   18.802117   
2  ecg_01_clone  7c:9e:bd:f6:ce:48               0             5   51.313273   
3  eeg_01_sybil  34:94:54:aa:79:e0               0             5  100.196625   
4        ecg_01  7c:9e:bd:f6:ce:40               1             6    9.268681   

   iat_mean   iat_std  seq_gap_mean  seq_gap_max  seq_reset_rate  \
0  0.100552  0.007834      1.000000

## Section 4: Extract Features and Prepare Data

In [ ]:
# Check which features are available in your dataset
print("="*70)
print("FEATURE COMPATIBILITY CHECK")
print("="*70)

available_features = [f for f in feature_names if f in df_test.columns]
missing_features = [f for f in feature_names if f not in df_test.columns]
extra_features = [f for f in df_test.columns if f not in feature_names and f.lower() not in ['label', 'target', 'class', 'node_id', 'mac']]

print(f"\n✓ Available features: {len(available_features)}/{len(feature_names)}")
for f in available_features[:10]:
    print(f"   - {f}")
if len(available_features) > 10:
    print(f"   ... and {len(available_features) - 10} more")

if missing_features:
    print(f"\n✗ Missing features ({len(missing_features)}):")
    for f in missing_features:
        print(f"   - {f}")
    print(f"\n⚠️  WARNING: These {len(missing_features)} features are missing!")
    print(f"   Missing features will be filled with 0 (neutral value)")
    print(f"   This may affect prediction accuracy!")

if extra_features:
    print(f"\nℹ Extra features in dataset (not used for prediction):")
    for f in extra_features[:5]:
        print(f"   - {f}")
    if len(extra_features) > 5:
        print(f"   ... and {len(extra_features) - 5} more")

print("\n" + "="*70)


In [ ]:
# Check if the dataset has a label column
has_label = 'label' in df_test.columns or 'target' in df_test.columns or 'Label' in df_test.columns

if has_label:
    # Find the label column
    label_col = [col for col in df_test.columns if col.lower() in ['label', 'target', 'class']]
    if label_col:
        label_col = label_col[0]
        y_true = df_test[label_col].values
        print(f"✓ Found label column: '{label_col}'")
        print(f"  - Class distribution:")
        unique, counts = np.unique(y_true, return_counts=True)
        for u, c in zip(unique, counts):
            pct = 100 * c / len(y_true)
            label_name = 'Normal' if u == 0 else 'Sybil'
            print(f"    - {label_name}: {c} samples ({pct:.1f}%)")
else:
    y_true = None
    print("⚠ No label column found. Will only show predictions without validation.")

# Select only the features that match training features
# Handle missing features gracefully
available_features = [f for f in feature_names if f in df_test.columns]
missing_features = [f for f in feature_names if f not in df_test.columns]

print(f"\n✓ Extracting {len(available_features)} available features...")

# Create features dataframe with available features
X_test = df_test[available_features].copy()

# Add missing features as zeros (neutral value)
if missing_features:
    print(f"⚠ Adding {len(missing_features)} missing features with value=0")
    for f in missing_features:
        X_test[f] = 0

# Reorder to match training feature order
X_test = X_test[feature_names]

# Handle missing values
missing_count = X_test.isna().sum().sum()
if missing_count > 0:
    print(f"⚠ Found {missing_count} missing values. Filling with 0...")
    X_test = X_test.fillna(0)

# Scale the features using the same scaler from training
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Data prepared for prediction")
print(f"  - Shape: {X_test_scaled.shape}")
print(f"  - Features scaled using training scaler")
print(f"  - Available: {len(available_features)}/{len(feature_names)}")
if missing_features:
    print(f"  - Missing: {len(missing_features)} (filled with 0)")

⚠ No label column found. Will only show predictions without validation.


KeyError: "['boot_id'] not in index"

## Section 5: Make Predictions on Real-World Data

In [ ]:
# Make predictions
print("Making predictions on real-world dataset...\n")

# Get predictions (0 = Normal, 1 = Sybil)
y_pred = rf_model.predict(X_test_scaled)

# Get prediction probabilities
y_prob = rf_model.predict_proba(X_test_scaled)
confidence = np.max(y_prob, axis=1)  # Confidence score (0-1)

print(f"✓ Predictions completed")
print(f"  - Samples predicted: {len(y_pred)}")

## Section 6: Detection Results Summary

In [ ]:
# Count detected nodes
n_normal = (y_pred == 0).sum()
n_sybil = (y_pred == 1).sum()
sybil_percentage = 100 * n_sybil / len(y_pred)
normal_percentage = 100 * n_normal / len(y_pred)

print("="*70)
print("SYBIL DETECTION RESULTS - REAL-WORLD WBAN DATA")
print("="*70)
print(f"\nTotal Nodes Tested: {len(y_pred)}")
print(f"\n📊 DETECTION SUMMARY:")
print(f"  ✓ Normal Nodes:  {n_normal:6d} ({normal_percentage:6.2f}%)")
print(f"  ⚠ Sybil Nodes:   {n_sybil:6d} ({sybil_percentage:6.2f}%)")
print(f"\n  Average Confidence: {confidence.mean():.4f}")
print(f"  Min Confidence:     {confidence.min():.4f}")
print(f"  Max Confidence:     {confidence.max():.4f}")
print("="*70)

## Section 7: List All Detected Nodes

In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'Sample_ID': range(len(y_pred)),
    'Prediction': ['Normal' if p == 0 else 'Sybil' for p in y_pred],
    'Sybil_Probability': y_prob[:, 1],  # Probability of being Sybil
    'Confidence': confidence
})

# Add true labels if available
if y_true is not None:
    results_df['True_Label'] = ['Normal' if y == 0 else 'Sybil' for y in y_true]
    results_df['Correct'] = (y_pred == y_true)

# Add node_id if available in original data
if 'node_id' in df_test.columns:
    results_df['Node_ID'] = df_test['node_id'].values
elif 'mac' in df_test.columns:
    results_df['MAC_Address'] = df_test['mac'].values

print("\n" + "="*70)
print("DETAILED NODE-BY-NODE RESULTS")
print("="*70)
print(results_df.to_string(index=False))
print("="*70)

## Section 8: Detailed Analysis - Sybil Nodes

In [ ]:
print("\n" + "="*70)
print("SYBIL NODES DETECTED (DETAILED)")
print("="*70)

sybil_nodes = results_df[results_df['Prediction'] == 'Sybil'].copy()
sybil_nodes = sybil_nodes.sort_values('Sybil_Probability', ascending=False)

if len(sybil_nodes) > 0:
    print(f"\nFound {len(sybil_nodes)} Sybil node(s):\n")
    for idx, row in sybil_nodes.iterrows():
        sample_id = row['Sample_ID']
        prob = row['Sybil_Probability']
        conf = row['Confidence']
        
        node_info = ""
        if 'Node_ID' in row:
            node_info = f" (Node ID: {row['Node_ID']})"
        elif 'MAC_Address' in row:
            node_info = f" (MAC: {row['MAC_Address']})"
        
        status = ""
        if 'True_Label' in results_df.columns and row['True_Label'] == 'Sybil':
            status = " ✓ CORRECT DETECTION"
        elif 'True_Label' in results_df.columns and row['True_Label'] == 'Normal':
            status = " ✗ FALSE POSITIVE"
        
        print(f"  [{sample_id:3d}] Sybil Probability: {prob:.4f} | Confidence: {conf:.4f}{node_info}{status}")
else:
    print("\n✓ No Sybil nodes detected!")

## Section 9: Detailed Analysis - Normal Nodes

In [ ]:
print("\n" + "="*70)
print("NORMAL NODES DETECTED (DETAILED)")
print("="*70)

normal_nodes = results_df[results_df['Prediction'] == 'Normal'].copy()
normal_nodes = normal_nodes.sort_values('Sybil_Probability', ascending=True)

if len(normal_nodes) > 0:
    print(f"\nFound {len(normal_nodes)} Normal node(s):\n")
    
    # Show top 10 most confident normal nodes
    top_normal = normal_nodes.nlargest(10, 'Confidence')
    
    for idx, row in top_normal.iterrows():
        sample_id = row['Sample_ID']
        prob = row['Sybil_Probability']
        conf = row['Confidence']
        
        node_info = ""
        if 'Node_ID' in row:
            node_info = f" (Node ID: {row['Node_ID']})"
        elif 'MAC_Address' in row:
            node_info = f" (MAC: {row['MAC_Address']})"
        
        status = ""
        if 'True_Label' in results_df.columns and row['True_Label'] == 'Normal':
            status = " ✓ CORRECT DETECTION"
        elif 'True_Label' in results_df.columns and row['True_Label'] == 'Sybil':
            status = " ✗ FALSE NEGATIVE"
        
        print(f"  [{sample_id:3d}] Sybil Prob: {prob:.4f} | Confidence: {conf:.4f}{node_info}{status}")
    
    if len(normal_nodes) > 10:
        print(f"\n  ... and {len(normal_nodes) - 10} more normal nodes")
else:
    print("\n⚠ All nodes detected as Sybil!")

## Section 10: Performance Metrics (If Labels Available)

In [ ]:
if y_true is not None:
    print("\n" + "="*70)
    print("PERFORMANCE METRICS ON REAL-WORLD DATA")
    print("="*70)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    try:
        roc_auc = roc_auc_score(y_true, y_prob[:, 1])
    except:
        roc_auc = None
    
    print(f"\n  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
    print(f"  F1-Score:  {f1:.4f}")
    if roc_auc is not None:
        print(f"  ROC-AUC:   {roc_auc:.4f}")
    
    # Confusion Matrix
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(f"  TN: {cm[0,0]:6d}  FP: {cm[0,1]:6d}")
    print(f"  FN: {cm[1,0]:6d}  TP: {cm[1,1]:6d}")
    
    # Classification Report
    print(f"\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Sybil']))
    
    print("="*70)
else:
    print("\n⚠ No true labels available. Cannot compute performance metrics.")

## Section 11: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Real-World WBAN Sybil Detection Results', fontsize=14, fontweight='bold')

# Plot 1: Detection Distribution
ax = axes[0, 0]
labels = ['Normal', 'Sybil']
sizes = [n_normal, n_sybil]
colors = ['#2ecc71', '#e74c3c']
explode = (0, 0.1) if n_sybil > 0 else (0, 0)
ax.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Detection Distribution')

# Plot 2: Confidence Distribution
ax = axes[0, 1]
ax.hist(confidence, bins=30, color='#3498db', alpha=0.7, edgecolor='black')
ax.axvline(confidence.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {confidence.mean():.3f}')
ax.set_xlabel('Confidence Score')
ax.set_ylabel('Frequency')
ax.set_title('Model Confidence Distribution')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Sybil Probability Distribution
ax = axes[1, 0]
sybil_probs = y_prob[:, 1]
ax.hist(sybil_probs, bins=30, color='#9b59b6', alpha=0.7, edgecolor='black')
ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_xlabel('Sybil Probability')
ax.set_ylabel('Frequency')
ax.set_title('Sybil Probability Distribution')
ax.legend()
ax.grid(alpha=0.3)

# Plot 4: Performance (if labels available)
ax = axes[1, 1]
if y_true is not None:
    metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    metrics_values = [accuracy, precision, recall, f1]
    colors_bar = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
    ax.bar(metrics_names, metrics_values, color=colors_bar, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Score')
    ax.set_title('Performance Metrics')
    ax.set_ylim([0, 1])
    for i, v in enumerate(metrics_values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No labels available\nfor metrics', ha='center', va='center', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.tight_layout()
plt.savefig('realworld_sybil_detection_results.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved as 'realworld_sybil_detection_results.png'")
plt.show()

## Section 12: Export Results to CSV

In [ ]:
# Save detailed results to CSV
output_csv = 'sybil_detection_results.csv'
results_df.to_csv(output_csv, index=False)
print(f"✓ Detailed results saved: {output_csv}")

# Save summary report
summary_report = f"""
SYBIL DETECTION SUMMARY REPORT
========================================
Date: {pd.Timestamp.now()}
Model: Random Forest (Stage 2)

RESULTS:
========
Total Nodes: {len(y_pred)}
Normal Nodes: {n_normal} ({normal_percentage:.2f}%)
Sybil Nodes: {n_sybil} ({sybil_percentage:.2f}%)

Confidence:
  - Average: {confidence.mean():.4f}
  - Min: {confidence.min():.4f}
  - Max: {confidence.max():.4f}

Dataset Features:
  - Available: {len(available_features)}/{len(feature_names)}
  - Missing: {len(missing_features)} (filled with 0)

"""

if y_true is not None:
    summary_report += f"""
PERFORMANCE METRICS:
====================
Accuracy: {accuracy:.4f}
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-Score: {f1:.4f}
"""

output_txt = 'sybil_detection_summary.txt'
with open(output_txt, 'w') as f:
    f.write(summary_report)
print(f"✓ Summary report saved: {output_txt}")

## Summary

This notebook has:
1. ✓ Loaded your trained Random Forest model
2. ✓ Processed your real-world WBAN test data
3. ✓ Made predictions on each node
4. ✓ Listed all Sybil nodes detected with confidence scores
5. ✓ Listed all Normal nodes
6. ✓ Generated performance metrics (if labels available)
7. ✓ Created visualizations
8. ✓ Exported results to CSV and text files

### Output Files:
- `sybil_detection_results.csv` - Detailed node-by-node results
- `sybil_detection_summary.txt` - Text summary
- `realworld_sybil_detection_results.png` - Visualization charts

### Next Steps:
1. Review the CSV results to see which specific nodes are flagged
2. Check visualizations for detection patterns
3. If performance is acceptable, deploy to production
4. Monitor false positives and adjust threshold if needed